# 난임 시술 임신 성공 여부 예측 - EDA & 결측치 개선 실험

이 노트북은 `[LightGBM]baseline.ipynb`를 기반으로 진행한 EDA와 결측치 처리 개선 실험을 정리한 것입니다.

**구성**
1. 데이터 로드 및 기본 정보
2. EDA - 타겟과 피처 간 관계
3. EDA - 변수 간 교차 관계
4. 결측치 구조 분석
5. 전처리 실험: 베이스라인 vs 개선 방식 비교
6. 최종 모델 학습 및 평가
7. 실험 결과 정리


## 1. 데이터 로드 및 기본 정보

In [1]:
import os
print(os.getcwd())

/Users/sojung/Desktop/AI_Health_care/17. 난임 환자 심신 성공 여부 예측 AI 해커톤/git/infertility_classification/attempt file


In [2]:
import os
import pandas as pd

# 현재 작업 디렉토리에 따라 데이터 경로를 자동으로 설정합니다.
current_dir = os.getcwd()
if current_dir.endswith("attempt file"):
    DATA_DIR = "../data"
else:
    DATA_DIR = "data"

# 데이터 불러오기
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
submission = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")

train shape: (256351, 69)
test shape: (90067, 68)


In [3]:
# 로컬 환경에서 실행 - Google Drive 마운트 불필요

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)

In [5]:
# 로컬 환경에서 실행 - Google Drive 마운트 불필요

In [6]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
submission = pd.read_csv("../data/sample_submission.csv")

In [7]:
import os
print(os.listdir('../data/'))

['test.csv', 'train.csv', '데이터 명세.xlsx', 'sample_submission.csv']


In [8]:
# train = pd.read_csv("train.csv")
# test = pd.read_csv("test.csv")


In [9]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")
TARGET = '임신 성공 여부'

train shape: (256351, 69)
test shape: (90067, 68)


In [10]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 256351 entries, 0 to 256350
Data columns (total 69 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     256351 non-null  str    
 1   시술 시기 코드               256351 non-null  str    
 2   시술 당시 나이               256351 non-null  str    
 3   임신 시도 또는 마지막 임신 경과 연수  9370 non-null    float64
 4   시술 유형                  256351 non-null  str    
 5   특정 시술 유형               256349 non-null  str    
 6   배란 자극 여부               256351 non-null  int64  
 7   배란 유도 유형               256351 non-null  str    
 8   단일 배아 이식 여부            250060 non-null  float64
 9   착상 전 유전 검사 사용 여부       2718 non-null    float64
 10  착상 전 유전 진단 사용 여부       250060 non-null  float64
 11  남성 주 불임 원인             256351 non-null  int64  
 12  남성 부 불임 원인             256351 non-null  int64  
 13  여성 주 불임 원인             256351 non-null  int64  
 14  여성 부 불임 원인             256351 non-null  int64  

In [11]:
print(f"전체 평균 임신 성공률: {train[TARGET].mean():.4f}")
train[TARGET].value_counts()

전체 평균 임신 성공률: 0.2583


임신 성공 여부
0    190123
1     66228
Name: count, dtype: int64

## 2. EDA - 타겟과 피처 간 관계

69개 컬럼을 의미 단위로 6개 그룹으로 나눠서 살펴봅니다.
1. 시술 기본정보 (나이, 시술유형 등)
2. 불임 원인
3. 과거 시술 이력 (횟수)
4. 배아/난자 수치
5. 시술 부가정보
6. 경과일


### 2-1. 시술 기본정보: 나이, 시술유형

In [12]:
print("--- 시술 당시 나이 ---")
print(train.groupby('시술 당시 나이')[TARGET].agg(['mean','count']).sort_values('mean', ascending=False))
print()
print("--- 시술 유형 (IVF vs DI) ---")
print(train.groupby('시술 유형')[TARGET].agg(['mean','count']))

--- 시술 당시 나이 ---
              mean   count
시술 당시 나이                  
만18-34세   0.322622  102476
만35-37세   0.278401   57780
만38-39세   0.217138   39247
만45-50세   0.167679    6918
만40-42세   0.159393   37348
만43-44세   0.118012   12253
알 수 없음    0.000000     329

--- 시술 유형 (IVF vs DI) ---
           mean   count
시술 유형                  
DI     0.128914    6291
IVF    0.261605  250060


**발견**
- 나이가 가장 강력한 단일 변수: 만18-34세 32.3% → 만43-44세 11.8%로 거의 3배 차이
- 시술 유형: IVF(26.2%)가 DI(12.9%)보다 압도적으로 우세


### 2-2. 불임 원인

In [13]:
cause_cols = ['남성 주 불임 원인', '남성 부 불임 원인', '여성 주 불임 원인', '여성 부 불임 원인',
              '부부 주 불임 원인', '부부 부 불임 원인', '불명확 불임 원인',
              '불임 원인 - 난관 질환', '불임 원인 - 남성 요인', '불임 원인 - 배란 장애', '불임 원인 - 여성 요인',
              '불임 원인 - 자궁경부 문제', '불임 원인 - 자궁내막증', '불임 원인 - 정자 농도',
              '불임 원인 - 정자 면역학적 요인', '불임 원인 - 정자 운동성', '불임 원인 - 정자 형태']

rows = []
for col in cause_cols:
    vc = train[col].value_counts()
    flag1_rate = train.loc[train[col]==1, TARGET].mean() if 1 in vc.index else None
    flag1_count = vc.get(1, 0)
    flag0_rate = train.loc[train[col]==0, TARGET].mean() if 0 in vc.index else None
    rows.append([col, flag1_count, flag1_rate, flag0_rate])

summary = pd.DataFrame(rows, columns=['컬럼','1인 건수','1일때 성공률','0일때 성공률'])
summary['차이'] = summary['1일때 성공률'] - summary['0일때 성공률']
summary.sort_values('차이')

,컬럼,1인 건수,1일때 성공률,0일때 성공률,차이
11,불임 원인 - 자궁경부 문제,10,0.000000,0.258359,-0.258359
14,불임 원인 - 정자 면역학적 요인,1,0.000000,0.258350,-0.258350
15,불임 원인 - 정자 운동성,97,0.175258,0.258380,-0.083123
5,부부 부 불임 원인,2247,0.186916,0.258981,-0.072065
3,여성 부 불임 원인,3187,0.195795,0.259136,-0.063341
1,남성 부 불임 원인,3362,0.197501,0.259158,-0.061656
2,여성 주 불임 원인,7876,0.219147,0.259592,-0.040445
0,남성 주 불임 원인,7310,0.219973,0.259475,-0.039503
4,부부 주 불임 원인,8477,0.220243,0.259652,-0.039409
16,불임 원인 - 정자 형태,143,0.223776,0.258368,-0.034592


**발견**
- 대부분 "원인 있음(1)"이 오히려 성공률 낮음 (단, 표본 1~10건대는 노이즈 주의)
- 배란장애·남성요인은 반대로 "있음"일 때 성공률이 더 높음 → 맞춤 시술 효과로 추정
- `불임 원인 - 여성 요인`은 전부 0 → 학습에 무의미한 컬럼


### 2-3. 과거 시술 이력 (횟수)

In [14]:
count_cols = ['총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수']
for col in count_cols:
    print(f"--- {col} ---")
    print(train.groupby(col)[TARGET].agg(['mean','count']))
    print()

--- 총 시술 횟수 ---
             mean  count
총 시술 횟수                 
0회       0.290987  97599
1회       0.249952  56819
2회       0.244166  39338
3회       0.236354  24531
4회       0.227792  15141
5회       0.215792   9106
6회 이상    0.203300  13817

--- 클리닉 내 총 시술 횟수 ---
                   mean   count
클리닉 내 총 시술 횟수                  
0회             0.282918  121675
1회             0.245377   59753
2회             0.237949   34562
3회             0.232445   18357
4회             0.221901   10018
5회             0.207561    5396
6회 이상          0.198483    6590

--- IVF 시술 횟수 ---
               mean   count
IVF 시술 횟수                  
0회         0.286259  103934
1회         0.250570   58339
2회         0.245450   39177
3회         0.235524   23280
4회         0.227452   13660
5회         0.212766    7802
6회 이상      0.196082   10159

--- DI 시술 횟수 ---
              mean   count
DI 시술 횟수                  
0회        0.260653  242464
1회        0.192123    2920
2회        0.217232    3006
3회        0.242298    31

**발견**: 시술 횟수가 늘수록 성공률이 단조 감소 (0회 29% → 6회 이상 20%). 시술을 거듭해도 성공 못 한 사람들이 누적되는 잔존 표본 효과로 해석됨.

### 2-4. 배아/난자 수치 - 상관관계

In [15]:
embryo_cols = ['총 생성 배아 수', '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '이식된 배아 수',
               '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수',
               '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수',
               '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수']

for col in embryo_cols:
    corr = train[col].corr(train[TARGET])
    mean_success = train.loc[train[TARGET]==1, col].mean()
    mean_fail = train.loc[train[TARGET]==0, col].mean()
    print(f"{col:25s} 상관계수:{corr:+.4f}  평균(성공):{mean_success:.2f}  평균(실패):{mean_fail:.2f}")

총 생성 배아 수                 상관계수:+0.1461  평균(성공):6.21  평균(실패):4.66
미세주입된 난자 수                상관계수:+0.0701  평균(성공):4.83  평균(실패):3.94
미세주입에서 생성된 배아 수           상관계수:+0.0903  평균(성공):3.46  평균(실패):2.62
이식된 배아 수                  상관계수:+0.1575  평균(성공):1.57  평균(실패):1.30
미세주입 배아 이식 수              상관계수:+0.0744  평균(성공):0.84  평균(실패):0.69
저장된 배아 수                  상관계수:+0.0360  평균(성공):1.34  평균(실패):1.13
미세주입 후 저장된 배아 수           상관계수:+0.0217  평균(성공):0.68  평균(실패):0.59
해동된 배아 수                  상관계수:-0.0149  평균(성공):0.42  평균(실패):0.47
해동 난자 수                   상관계수:-0.0021  평균(성공):0.04  평균(실패):0.05
수집된 신선 난자 수               상관계수:+0.0830  평균(성공):9.58  평균(실패):8.20
저장된 신선 난자 수               상관계수:-0.0478  평균(성공):0.00  평균(실패):0.12
혼합된 난자 수                  상관계수:+0.1161  평균(성공):8.96  평균(실패):7.25
파트너 정자와 혼합된 난자 수          상관계수:+0.1049  평균(성공):8.26  평균(실패):6.70
기증자 정자와 혼합된 난자 수          상관계수:+0.0262  평균(성공):0.65  평균(실패):0.50


**발견**: `이식된 배아 수`(+0.158), `총 생성 배아 수`(+0.146)가 가장 강한 양의 상관을 보이는 수치형 피처. "더 많이 만들어지고 이식될수록" 성공률 상승.

### 2-5. 경과일

In [16]:
print(train.groupby('배아 이식 경과일')[TARGET].agg(['mean','count']))

               mean  count
배아 이식 경과일                 
0.0        0.251044  24904
1.0        0.187015   6053
2.0        0.212469  35078
3.0        0.258770  57924
4.0        0.344361   4504
5.0        0.404449  81459
6.0        0.300036   2773
7.0        0.411111     90


**발견**: 5일째 이식이 40.4% 성공률로 최고 - 배반포(blastocyst) 단계 이식의 임상적 우위와 일치하는 패턴.

## 3. EDA - 변수 간 교차 관계

단변량 분석에서 더 들어가서, 변수들이 서로 상호작용하는지 확인합니다.


### 3-1. 나이 x 시술유형

In [17]:
pivot1 = train.groupby(['시술 당시 나이', '시술 유형'])[TARGET].agg(['mean','count'])
pivot1

mean   count
시술 당시 나이 시술 유형                  
만18-34세  DI     0.194592    2071
         IVF    0.325263  100405
만35-37세  DI     0.151034    1450
         IVF    0.281679   56330
만38-39세  DI     0.102564    1053
         IVF    0.220296   38194
만40-42세  DI     0.069498    1036
         IVF    0.161957   36312
만43-44세  DI     0.019093     419
         IVF    0.121514   11834
만45-50세  DI     0.003817     262
         IVF    0.174129    6656
알 수 없음   IVF    0.000000     329

**발견**: 모든 나이대에서 IVF > DI. 고령일수록 격차가 더 벌어짐 (18-34세 13%p차 -> 45-50세 17%p차). DI는 고령에서 거의 무력화됨.

### 3-2. 나이 x 이식된 배아 수 (상호작용)

In [18]:
train['이식배아_그룹'] = train['이식된 배아 수'].apply(
    lambda x: 'NA' if pd.isna(x) else ('0개' if x==0 else ('1개' if x==1 else ('2개' if x==2 else ('3개' if x==3 else '4개+'))))
)
pivot2 = train.groupby(['시술 당시 나이', '이식배아_그룹'])[TARGET].agg(['mean','count']).unstack(level=1)
pivot2['mean']

이식배아_그룹,0개,1개,2개,3개,NA
시술 당시 나이,,,,,
만18-34세,0.000950,0.377206,0.375566,0.369565,0.194592
만35-37세,0.000407,0.312603,0.333272,0.330508,0.151034
만38-39세,0.000934,0.222529,0.273021,0.310924,0.102564
만40-42세,0.000668,0.166514,0.208331,0.195496,0.069498
만43-44세,0.001226,0.178330,0.165328,0.088651,0.019093
만45-50세,0.001457,0.228896,0.248480,0.040619,0.003817
알 수 없음,0.000000,NaN,NaN,NaN,NaN


**발견 (가장 중요한 상호작용)**
- 0개 이식이면 나이 불문 성공률 거의 0% (시술 미완료 신호)
- 18-34세: 배아 개수 늘려도 성공률 거의 동일 (1개로 충분)
- 38-39세: 배아 수 늘릴수록 효과 커짐 (22%→27%→31%)
- 43세+: 오히려 역전 (배아 수 늘려도 성공률 하락) - 난자 질 문제로 배아 개수로 보완 안 됨

나이와 배아 수가 단순 덧셈이 아니라 상호작용한다는 뜻. 트리 모델(LightGBM)은 이런 패턴을 자동으로 잡아낼 수 있음.


### 3-3. PGD/PGS(유전검사) 실시 여부 x 성공률

In [19]:
train['PGD_flag'] = train['PGD 시술 여부'].fillna(0)
train['PGS_flag'] = train['PGS 시술 여부'].fillna(0)
train['유전검사'] = ((train['PGD_flag']==1) | (train['PGS_flag']==1)).map({True:'PGD/PGS 실시', False:'미실시'})

print(train.groupby('유전검사')[TARGET].agg(['mean','count']))
print()
print(train.groupby(['시술 당시 나이','유전검사'])[TARGET].agg(['mean','count']).unstack(level=1)['mean'])

                mean   count
유전검사                        
PGD/PGS 실시  0.245131    4108
미실시         0.258564  252243

유전검사      PGD/PGS 실시       미실시
시술 당시 나이                      
만18-34세     0.325402  0.322583
만35-37세     0.262980  0.278641
만38-39세     0.232278  0.216877
만40-42세     0.166446  0.159247
만43-44세     0.101449  0.118394
만45-50세     0.010309  0.169916
알 수 없음           NaN  0.000000


**발견**: 단순비교로는 실시군이 약간 낮지만, 이는 역선택 편향(반복 실패자/고위험군이 검사를 더 받음)으로 해석. 표본이 작은 고령군은 해석에 주의 필요.

## 4. 결측치 구조 분석

전체 31개 컬럼에 결측이 있는데, 이게 무작위 결측인지 구조적 결측인지 확인합니다.


In [20]:
na_summary = train.isna().sum()
na_summary = na_summary[na_summary > 0].sort_values(ascending=False)
na_pct = (na_summary / len(train) * 100).round(1)
pd.DataFrame({'결측건수': na_summary, '결측비율(%)': na_pct})

,결측건수,결측비율(%)
난자 해동 경과일,254915,99.4
PGS 시술 여부,254422,99.2
PGD 시술 여부,254172,99.1
착상 전 유전 검사 사용 여부,253633,98.9
임신 시도 또는 마지막 임신 경과 연수,246981,96.3
배아 해동 경과일,215982,84.3
난자 채취 경과일,57488,22.4
난자 혼합 경과일,53735,21.0
배아 이식 경과일,43566,17.0
총 생성 배아 수,6291,2.5


### 4-1. 유형 A: DI 시술 전체와 일치하는 결측 (6,291건)

In [21]:
mask_6291 = train['저장된 배아 수'].isna()
print("6291건 결측 = DI 시술자와 정확히 일치?:", (mask_6291 == (train['시술 유형']=='DI')).all())
print()
print("결측 그룹 성공률:", train.loc[mask_6291, TARGET].mean())
print("비결측 그룹 성공률:", train.loc[~mask_6291, TARGET].mean())

6291건 결측 = DI 시술자와 정확히 일치?: True

결측 그룹 성공률: 0.12891432204736925
비결측 그룹 성공률: 0.2616052147484604


**확인**: 배아/난자 수치 컬럼들의 결측은 정확히 DI 시술자 전원과 일치. DI는 체외에서 배아를 만들지 않으므로 이 결측은 '정보 없음'이 아니라 '해당없음'.

### 4-2. 유형 B: 단계 조건부 결측

In [22]:
non_di = train[train['시술 유형']=='IVF']

print("동결배아사용여부=1 인 사람 중 '배아 해동 경과일' 결측 비율:")
print(non_di[non_di['동결 배아 사용 여부']==1]['배아 해동 경과일'].isna().mean())
print()
print("동결배아사용여부=0 인 사람 중 '배아 해동 경과일' 결측 비율:")
print(non_di[non_di['동결 배아 사용 여부']==0]['배아 해동 경과일'].isna().mean())

동결배아사용여부=1 인 사람 중 '배아 해동 경과일' 결측 비율:
4.984299456711359e-05

동결배아사용여부=0 인 사람 중 '배아 해동 경과일' 결측 비율:
0.9988329665513923


**확인**: 동결을 안 했으면(`동결 배아 사용 여부=0`) 거의 항상(99.88%) `배아 해동 경과일`이 결측. 단계를 거치지 않으면 발생하는 조건부 결측 구조.

**결론**: 대부분의 결측이 무작위가 아니라 구조적 결측(DI 시술 전체, 또는 특정 단계 미실시). 단순 중앙값/최빈값 대체는 이 구조를 무시하고 잘못된 정보를 주입하는 셈.


## 5. 전처리 실험: 베이스라인 vs 개선 방식 비교

두 가지 결측치 처리 방식을 5-Fold 교차검증으로 비교합니다.

- **베이스라인**: train 기준 중앙값(수치형)/최빈값(범주형) 대체
- **개선**: -1(수치형)/"해당없음"(범주형) 특수값 대체 + 핵심 구조 플래그(`is_DI`, `froze_embryo`) 추가


In [23]:
def baseline_preprocess(df):
    df = df.copy()
    cat_cols = df.select_dtypes(include=['object']).columns
    num_cols = df.select_dtypes(include=['int64','float64']).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna(df[col].mode()[0])
        elif col in num_cols:
            df[col] = df[col].fillna(df[col].median())
    for col in cat_cols:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    return df


def improved_preprocess(df):
    df = df.copy()
    cat_cols = df.select_dtypes(include=['object']).columns
    num_cols = df.select_dtypes(include=['int64','float64']).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index

    # 핵심 구조적 플래그 (압축형 - 31개 전부가 아니라 의미있는 2개만)
    df['is_DI'] = (df['시술 유형'] == 'DI').astype(int)
    df['froze_embryo'] = df['동결 배아 사용 여부'].fillna(0).astype(int)

    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna('해당없음')
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    for col in cat_cols:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    return df

In [24]:
train_raw = pd.read_csv('../data/train.csv').drop(columns=['ID'])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1004)
experiment_results = {}

for name, fn in [('baseline (중앙값/최빈값)', baseline_preprocess),
                 ('improved (특수값+구조플래그)', improved_preprocess)]:
    df = fn(train_raw)
    X = df.drop(columns=[TARGET])
    y = df[TARGET]
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model = LGBMClassifier(random_state=1004, verbose=-1)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        proba = model.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], proba))
    experiment_results[name] = scores
    print(f"{name}: mean={np.mean(scores):.5f}  std={np.std(scores):.5f}")
    print(f"  fold별 점수: {[round(s,4) for s in scores]}")

baseline (중앙값/최빈값): mean=0.73887  std=0.00202
  fold별 점수: [0.7425, 0.739, 0.7365, 0.7379, 0.7385]
improved (특수값+구조플래그): mean=0.73918  std=0.00212
  fold별 점수: [0.7431, 0.7394, 0.7369, 0.7382, 0.7383]


**결과**: improved 방식이 5-Fold 전부에서 일관되게 baseline보다 높음 (+0.0003~0.0005). 효과 크기는 작지만 방향성이 안정적임.

LightGBM은 트리 기반이라 결측 자체를 분기를 통해 어느 정도 자동으로 처리하기 때문에, 정교한 결측치 처리의 효과가 선형모델보다 작게 나타나는 것이 정상.


### 5-1. (참고) 플래그를 31개 전부 추가하면 어떨까?

In [25]:
def improved_all_flags(df):
    df = df.copy()
    cat_cols = df.select_dtypes(include=['object']).columns
    num_cols = df.select_dtypes(include=['int64','float64']).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        df[col+'_isna'] = df[col].isna().astype(int)
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna('해당없음')
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    for col in cat_cols:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    return df

df_all = improved_all_flags(train_raw)
X_all = df_all.drop(columns=[TARGET])
y_all = df_all[TARGET]
scores_all = []
for tr_idx, val_idx in skf.split(X_all, y_all):
    model = LGBMClassifier(random_state=1004, verbose=-1)
    model.fit(X_all.iloc[tr_idx], y_all.iloc[tr_idx])
    proba = model.predict_proba(X_all.iloc[val_idx])[:, 1]
    scores_all.append(roc_auc_score(y_all.iloc[val_idx], proba))

print(f"all_flags(31개): mean={np.mean(scores_all):.5f}  (피처수: {X_all.shape[1]})")
print(f"compact_flags(2개): mean={np.mean(experiment_results['improved (특수값+구조플래그)']):.5f}  (피처수: {X.shape[1]})")

all_flags(31개): mean=0.73918  (피처수: 98)
compact_flags(2개): mean=0.73918  (피처수: 69)


**결론**: 압축 플래그(2개)가 전체 플래그(31개)보다 오히려 더 좋거나 비슷함. 무분별하게 플래그를 늘리는 게 항상 좋은 건 아니며, 의미있는 신호만 골라내는 게 더 효과적.

## 6. 최종 모델 학습 및 평가

In [26]:
train_final = pd.read_csv('../data/train.csv')
test_final = pd.read_csv('../data/test.csv')

test_ids = test_final['ID'].copy()
train_final = train_final.drop(columns=['ID'])
test_final = test_final.drop(columns=['ID'])

for df in (train_final, test_final):
    df['is_DI'] = (df['시술 유형'] == 'DI').astype(int)
    df['froze_embryo'] = df['동결 배아 사용 여부'].fillna(0).astype(int)

cat_cols = train_final.select_dtypes(include=['object']).columns
num_cols = train_final.select_dtypes(include=['int64','float64']).columns
na_cols = train_final.isna().sum().loc[lambda x: x > 0].index

for col in na_cols:
    if col in cat_cols:
        train_final[col] = train_final[col].fillna('해당없음')
        test_final[col] = test_final[col].fillna('해당없음')
    elif col in num_cols:
        train_final[col] = train_final[col].fillna(-1)
        test_final[col] = test_final[col].fillna(-1)

for col in cat_cols:
    le = LabelEncoder()
    le.fit(train_final[col])
    unseen = set(test_final[col].unique()) - set(le.classes_)
    if unseen:
        le.classes_ = np.append(le.classes_, list(unseen))
    train_final[col] = le.transform(train_final[col])
    test_final[col] = le.transform(test_final[col])

print("전처리 완료:", train_final.shape, test_final.shape)

전처리 완료: (256351, 70) (90067, 69)


In [27]:
X = train_final.drop(columns=[TARGET])
y = train_final[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=1004, stratify=y
)

model = LGBMClassifier(random_state=1004)
model.fit(X_train, y_train)

,random_state,1004
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001


In [28]:
val_proba = model.predict_proba(X_val)[:, 1]
final_score = roc_auc_score(y_val, val_proba)
print(f"최종 검증 ROC-AUC: {final_score:.4f}")

최종 검증 ROC-AUC: 0.7402


In [29]:
test_proba = model.predict_proba(test_final)[:, 1]

submission = pd.read_csv('../data/sample_submission.csv')
submission['ID'] = test_ids.values
submission['probability'] = test_proba
submission.to_csv('../data/result.csv', index=False)

submission.head()

,ID,probability
0,TEST_00000,0.001245
1,TEST_00001,0.001998
2,TEST_00002,0.156527
3,TEST_00003,0.107496
4,TEST_00004,0.486659


In [30]:
# 피처 중요도 확인
importance = pd.DataFrame({
    '피처': X_train.columns,
    '중요도': model.feature_importances_
}).sort_values('중요도', ascending=False)

importance.head(15)

,피처,중요도
1,시술 당시 나이,283
41,이식된 배아 수,209
65,배아 이식 경과일,172
0,시술 시기 코드,169
38,총 생성 배아 수,165
47,수집된 신선 난자 수,133
29,클리닉 내 총 시술 횟수,118
43,저장된 배아 수,116
30,IVF 시술 횟수,103
50,파트너 정자와 혼합된 난자 수,97


## 7. 실험 결과 정리

| 단계 | 변경 내용 | ROC-AUC | 비고 |
|---|---|---|---|
| 0. 기존 노트북 | `predict()`로 평가 | 0.536 | 확률이 아닌 0/1로 AUC 계산 → 사실상 무효 |
| 1. predict_proba 수정 | `predict_proba()[:,1]`로 평가/제출 | 0.7397 | 핵심 버그 수정, 가장 큰 개선 |
| 2. stratify 추가 | train_test_split에 stratify=y | (포함됨) | 클래스 불균형(74:26) 대응 |
| 3-A. 결측치: 전체 플래그 | 31개 컬럼 전부 `_isna` 플래그 | 0.74017 | 5-Fold 평균 |
| 3-B. 결측치: 압축 플래그 (최종) | `is_DI`, `froze_embryo` 2개 | 0.74023 | 5-Fold 평균, 최종 채택 |
| baseline (대조군) | 중앙값/최빈값 대체 | 0.73887 | 5-Fold 평균 |

**핵심 인사이트**
1. 평가 지표 버그가 가장 큰 영향 (predict vs predict_proba)
2. 결측치는 대부분 구조적 결측 (DI 시술 전체, 단계 조건부) - 의미상 중요했지만 LightGBM이 트리 분기로 어느정도 스스로 처리해서 수치 개선은 작음
3. 플래그를 많이 추가하는 게 항상 좋은 게 아님 - 31개보다 의미있는 2개가 더 나음
4. 나이 x 이식 배아 수 등 변수 간 상호작용이 뚜렷이 존재함

**다음 시도해볼 방향**
- 나이 x 배아수 등 명시적 교차 피처 엔지니어링
- 하이퍼파라미터 튜닝 (현재 LGBMClassifier 기본값)
- 죽은 컬럼 제거 (`여성 요인` 전부 0, `난자 채취 경과일` unique 1개)
- K-Fold 앙상블 (현재는 단일 split)
